In [ ]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.xgboost
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score
from imblearn.over_sampling import SMOTE
mlflow.set_tracking_uri("../mlruns")
# Load data
df = pd.read_csv('../data/train_featured.csv')

X = df.drop(columns=['isFraud', 'TransactionID'])
y = df['isFraud']

X = X.fillna(0)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# SMOTE
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print("Data ready.")
print(f"Train shape: {X_train_balanced.shape}")

c:\Users\phani\anaconda3\envs\sentinelscore\lib\site-packages\mlflow\utils\requirements_utils.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources  # noqa: TID251


Data ready.
Train shape: (911804, 218)


In [2]:
# Set experiment name
mlflow.set_experiment("sentinelscore-fraud-detection")

# Define parameters
params = {
    "n_estimators": 100,
    "max_depth": 6,
    "learning_rate": 0.1,
    "random_state": 42
}

# Start experiment run
with mlflow.start_run(run_name="baseline_xgboost"):
    
    # Train model
    model = XGBClassifier(**params, eval_metric='auc', n_jobs=-1)
    model.fit(X_train_balanced, y_train_balanced)
    
    # Evaluate
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    
    auc = roc_auc_score(y_test, y_pred_proba)
    f1 = f1_score(y_test, y_pred)
    
    # Log everything to MLflow
    mlflow.log_params(params)
    mlflow.log_metric("auc_roc", auc)
    mlflow.log_metric("f1_score", f1)
    mlflow.xgboost.log_model(model, "model")
    
    print(f"AUC-ROC: {auc:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print("Run logged to MLflow.")

2026/03/26 21:30:45 INFO mlflow.tracking.fluent: Experiment with name 'sentinelscore-fraud-detection' does not exist. Creating a new experiment.


c:\Users\phani\anaconda3\envs\sentinelscore\lib\site-packages\xgboost\sklearn.py:1116: UserWarning: [21:31:20] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1573: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between formats.
  self.get_booster().save_model(fname)


AUC-ROC: 0.8774
F1 Score: 0.5047
Run logged to MLflow.
